<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Validation strategy

**[C05-M04-L02 · Validation strategy](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/)** · C05, Machine Learning: From Problem to Reliable Model · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** use cross-validation on the train set to choose settings, keep the validation month for checks after the policy change, choose with the one-standard-error rule, and look at the test set once with `evaluate.py`.

| | |
|---|---|
| **Time** | About 70 minutes. The grid search takes about 15 seconds. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Average precision, from [Choose metrics](https://learning.nextia-ai.com/courses/ml/m04/choose-metrics/). Cross-validation in its simple form, from [Trees and ensembles](https://learning.nextia-ai.com/courses/ml/m03/trees-and-ensembles/). |
| **You do not need** | The local project. The setup below downloads the data, `ticket_model.py` and `train.py`. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M04-L02-validation-strategy/validation-strategy-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the C05 practice data (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m01/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

The cell also downloads the course's finished project files `ticket_model.py`, `train.py` from [C05/project](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C05/project) into `ticket-model/`. `ticket_model.py` is the final 11-feature version. `train.py` trains the model and chooses the threshold; you write it yourself in Package artifacts (Module 6). Section 6 runs it.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = ["ticket_model.py", "train.py"]
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Folds on 20 rows

**Cross-validation** (CV) tests a setting on every part of the train set in turn. **5-fold CV** cuts the rows into 5 parts, the **folds**. It fits the pipeline 5 times: each time on 4 folds, and it scores the fifth. Every row is scored once, by a model that did not see it. You get 5 scores: their mean and their spread.

Here are 20 rows. 5 of them escalated, and they are at the top, as in a file sorted by the target.

> **Predict.** Plain `KFold(5)` cuts the rows in order: rows 0–3, 4–7, and so on. How many escalations land in each fold? Then run the cell.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

y_20 = np.array([1] * 5 + [0] * 15)
x_20 = np.arange(20).reshape(-1, 1)
for name, folds in [("KFold", KFold(5)), ("StratifiedKFold", StratifiedKFold(5))]:
    print(name, "rows:", [rows.tolist() for _, rows in folds.split(x_20, y_20)][:2], "...")
    print(name, "escalated per fold:", [int(y_20[rows].sum()) for _, rows in folds.split(x_20, y_20)])

> **Check.** You should see `KFold escalated per fold: [4, 1, 0, 0, 0]` and `StratifiedKFold escalated per fold: [1, 1, 1, 1, 1]`.

Three folds of plain `KFold` have no escalation, so their average precision cannot be computed. **Stratified** CV keeps the share of escalations the same in every fold. Use `StratifiedKFold` for a classifier, and `shuffle=True` with a fixed `random_state` when the rows have an order.

## 2. CV on the real train set

The course always uses `StratifiedKFold(5, shuffle=True, random_state=0)` and average precision. Run the cell. It fits the final pipeline 5 times.

In [ ]:
from sklearn.model_selection import cross_val_score
from ticket_model import FEATURES, TARGET, build_pipeline, load

train, valid = load("train"), load("valid")
cv = StratifiedKFold(5, shuffle=True, random_state=0)
fold_scores = cross_val_score(build_pipeline(), train[FEATURES], train[TARGET], cv=cv, scoring="average_precision")
print("AP per fold:", fold_scores.round(3))
print(f"mean {fold_scores.mean():.4f}, std {fold_scores.std():.4f}, standard error {fold_scores.std() / 5 ** 0.5:.4f}")
print("escalated per fold:", [int(train[TARGET].iloc[rows].sum()) for _, rows in cv.split(train, train[TARGET])])

> **Check.** You should see `AP per fold: [0.325 0.321 0.347 0.364 0.34 ]`, then `mean 0.3393, std 0.0156, standard error 0.0070`, and 396 or 397 escalated per fold.

The same model scores from 0.321 to 0.364, only because the folds differ. The **standard error** (std ÷ √5 = 0.0070) is the typical error of the mean. Two settings whose means differ by less than one standard error are a tie.

## 3. Why the validation month still matters

CV mixes tickets from all 22 months of train. All of them are from before the warranty policy of 2026-05-01. The validation set is May 2026, after it.

> **Predict.** The model's mean score is its average guess of the escalation rate. Will it match the real rate on train? On May? Then run the cell.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof = cross_val_predict(build_pipeline(), train[FEATURES], train[TARGET], cv=cv, method="predict_proba")[:, 1]
pipeline = build_pipeline().fit(train[FEATURES], train[TARGET])
scores = pipeline.predict_proba(valid[FEATURES])[:, 1]
print(f"train, out of fold: mean score {oof.mean():.3f}, escalation rate {train[TARGET].mean():.3f}")
print(f"valid (May 2026):   mean score {scores.mean():.3f}, escalation rate {valid[TARGET].mean():.3f}")
monthly = train.groupby(train["created_at"].dt.strftime("%Y-%m"))[TARGET].mean()
print(f"train months: escalation rate from {monthly.min():.3f} to {monthly.max():.3f}")

> **Check.** You should see `mean score 0.117, escalation rate 0.117` on train, `mean score 0.120, escalation rate 0.155` on valid, and train months from `0.095` to `0.139`.

`cross_val_predict` gives each train ticket a score from the fold model that did not see it: an **out-of-fold** score. On train, the mean score matches the rate. In May, the rate (15.5%) is above every train month, and the model does not know it. CV cannot show this, because every fold is from before the change. So: **CV on train chooses the settings; the validation month checks the chosen model in the new conditions** and sets the threshold (Lesson 3).

## 4. GridSearchCV: ten settings, the same CV

A **hyperparameter** is a setting that you choose before fitting, such as `C` of logistic regression or the depth of the trees. `GridSearchCV` runs the same CV for every setting in a grid and ranks them. Here the grid has 4 logistic regressions and 6 gradient-boosting models: 10 settings × 5 folds = 50 fits.

Run the cell. It takes about 15 seconds.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

grid = [
    {"model": [LogisticRegression(max_iter=1000)], "model__C": [0.01, 0.1, 1, 10]},
    {"model": [HistGradientBoostingClassifier(random_state=0)],
     "model__max_depth": [2, 3, None], "model__learning_rate": [0.05, 0.1]},
]
search = GridSearchCV(build_pipeline(), grid, cv=cv, scoring="average_precision", n_jobs=-1)
search.fit(train[FEATURES], train[TARGET])
print("settings:", len(search.cv_results_["params"]))

> **Check.** You should see `settings: 10`.

The next cell turns the results into a table, best first. `describe()` gives each setting a short name. Run it.

In [ ]:
def describe(params):
    """A short name for one setting of the grid."""
    if isinstance(params["model"], LogisticRegression):
        return f"logistic, C={params['model__C']}"
    return f"boosting, rate {params['model__learning_rate']}, depth {params['model__max_depth']}"

results = pd.DataFrame(search.cv_results_)
results["setting"] = results["params"].map(describe)
results["se"] = results["std_test_score"] / 5 ** 0.5
table = results[["setting", "mean_test_score", "std_test_score", "se", "rank_test_score"]].sort_values("rank_test_score")
print(table.round(4).to_string(index=False))

> **Check.** The first row is `boosting, rate 0.1, depth 2` with 0.3424 and a standard error of 0.0080. Then `logistic, C=10` 0.3394, `logistic, C=1` 0.3393 and `logistic, C=0.1` 0.3391. The two boosting settings with depth `None` are last (0.3212 and 0.3188).

## 5. The one-standard-error rule

The best setting wins by 0.0031 over logistic regression with C = 1. Is that a real difference? The **one-standard-error rule**: take the best mean, subtract its standard error, and call every setting above that limit a tie with the best. From the tied settings, choose the simplest one.

> **Your turn.** Compute `limit` (the best mean minus its standard error), then `tied`: the list of settings in `table` whose mean is at or above `limit`. Run the check cell.

In [ ]:
best = table.iloc[0]
limit = best["mean_test_score"] - best["se"]
tied = table.loc[table["mean_test_score"] >= limit, "setting"].tolist()
print(round(limit, 4), tied)

In [ ]:
expect_hash('the tied settings', sorted(tied) if isinstance(tied, list) else tied, '16834725306e3ffa')

Eight settings tie with the best: all four logistic regressions and four boosting models. Only the boosting models without a depth limit are clearly worse. Priya chooses **logistic regression with the default C = 1**: it ties with the best, it is the simplest, its folds vary least (std 0.0156 against 0.0178), and its coefficients can be explained (Module 5). All four C values are within 0.002: tuning C does not matter here.

## 6. Test once with evaluate.py

The settings are chosen. Lesson 3 chooses the threshold on valid; `train.py` already does it with the same rule (0.19). Now, and only now, Priya looks at the test set: once, with a script that refuses to run twice.

Run the next cell. It writes `evaluate.py`, the course's file.

In [ ]:
%%writefile evaluate.py
"""Report the saved model's results on the test set, once.

Nextia Learning, C05, Module 4 (Validation strategy) and Module 6.

    python evaluate.py

Uses the threshold that train.py chose on valid. Writes
reports/test_report.json. If the report exists already, it stops: look at
the test set once, and do not change the model after you look.
"""

import json
import sys
from pathlib import Path

import joblib
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, roc_auc_score

from ticket_model import FEATURES, TARGET, load

REPORT = Path("reports/test_report.json")


def main():
    if REPORT.exists() and "--again" not in sys.argv:
        sys.exit(f"{REPORT} exists: the test set was used already. Read the report. "
                 "Run with --again only to reproduce it, never to choose a change.")
    pipeline = joblib.load("model/escalation_model.joblib")
    info = json.loads(Path("model/model_info.json").read_text(encoding="utf-8"))
    test = load("test")
    scores = pipeline.predict_proba(test[FEATURES])[:, 1]
    flags = (scores >= info["threshold"]).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(test[TARGET], flags, labels=[0, 1]).ravel())
    rule = (test["priority"] == 1).astype(int)
    rule_tp = int((rule & test[TARGET]).sum())
    report = {
        "rows": len(test), "escalated": int(test[TARGET].sum()), "threshold": info["threshold"],
        "roc_auc": round(roc_auc_score(test[TARGET], scores), 3),
        "average_precision": round(average_precision_score(test[TARGET], scores), 3),
        "brier": round(brier_score_loss(test[TARGET], scores), 4),
        "mean_score": round(float(scores.mean()), 3), "escalated_rate": round(float(test[TARGET].mean()), 3),
        "confusion": {"tp": tp, "fp": fp, "fn": fn, "tn": tn},
        "flag_rate": round((tp + fp) / len(test), 3),
        "precision": round(tp / (tp + fp), 3), "recall": round(tp / (tp + fn), 3),
        "baseline_priority_1": {"flagged": int(rule.sum()), "flag_rate": round(float(rule.mean()), 3),
                                "precision": round(rule_tp / rule.sum(), 3),
                                "recall": round(rule_tp / test[TARGET].sum(), 3)},
    }
    REPORT.parent.mkdir(exist_ok=True)
    REPORT.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

> **Check.** You should see `Writing evaluate.py` (or `Overwriting`).

`evaluate.py` needs the saved model. Run `train.py` first. `!"{sys.executable}"` runs a script with the notebook's own Python, like `python train.py` in a terminal.

In [ ]:
import sys
!"{sys.executable}" train.py

> **Check.** The first line is `Trained on 16939 tickets. Threshold 0.19 for a review capacity of 20%.` The second line shows the valid results: ROC AUC 0.752, average precision 0.391, flagged 227 of 1179 (19.3%), precision 0.379, recall 0.47.

> **Warning.** The next cell uses the test set. Run it once. Read the result, and do not change the model, the features or the threshold after it.

In [ ]:
!"{sys.executable}" evaluate.py

> **Check.** You should see a JSON report with `"roc_auc": 0.749`, `"average_precision": 0.345`, `"brier": 0.1207`, the confusion counts `"tp": 70, "fp": 126, "fn": 87, "tn": 719`, `"flag_rate": 0.196`, `"precision": 0.357`, `"recall": 0.446`, and the rule's `"precision": 0.367, "recall": 0.35`.

If you run this notebook a second time in the same folder, the cell above prints the guard message instead. That is correct: the report from your first run is in `reports/test_report.json`.

Now run `evaluate.py` again, as Tomás would after a small change.

In [ ]:
!"{sys.executable}" evaluate.py

> **Check.** You should see `reports/test_report.json exists: the test set was used already. Read the report. Run with --again only to reproduce it, never to choose a change.`

The guard is a reminder, not a lock: `--again` reproduces the same report. Read the report from the file:

In [ ]:
import json
from pathlib import Path

report = json.loads(Path("reports/test_report.json").read_text())
print({key: report[key] for key in ["roc_auc", "average_precision", "flag_rate", "precision", "recall"]})

In [ ]:
expect('the test average precision', report["average_precision"], 0.345)

The test AP (0.345) is lower than valid (0.391). That is normal: valid helped to choose, so it is a little optimistic, and June is a new month. The model still meets the improvement target on test: 19.6% flagged, recall 0.446 (the rule: 0.350). **We do not change anything after we look.**

## 7. Misconception: tune on the test set

Tomás says: "Let us try the 10 settings on the test set and keep the best." What would that do? To keep the real test set clean, Priya simulates it inside May. She cuts the May tickets at random into two halves. Half A plays the test set that Tomás tunes on. Half B plays new tickets. She repeats it 200 times.

The next cell fits the 10 settings on train and scores May. Run it. It takes about 10 seconds.

In [ ]:
from sklearn.base import clone

may_scores = {}
for params in search.cv_results_["params"]:
    settings = {key.split("__")[1]: value for key, value in params.items() if "__" in key}
    model = clone(params["model"]).set_params(**settings)
    fitted = build_pipeline(model).fit(train[FEATURES], train[TARGET])
    may_scores[describe(params)] = fitted.predict_proba(valid[FEATURES])[:, 1]
print(len(may_scores), "settings scored")

> **Check.** You should see `10 settings scored`.

> **Predict.** Tomás reports the AP of the best setting on half A. Will that setting score as well on half B? Then run the cell.

In [ ]:
from sklearn.metrics import average_precision_score

truth = valid[TARGET].to_numpy()
rng = np.random.default_rng(0)
reported, on_new, fixed = [], [], []
for draw in range(200):
    order = rng.permutation(len(truth))
    half_a, half_b = order[:589], order[589:]
    ap_a = {name: average_precision_score(truth[half_a], s[half_a]) for name, s in may_scores.items()}
    chosen = max(ap_a, key=ap_a.get)
    reported.append(ap_a[chosen])
    on_new.append(average_precision_score(truth[half_b], may_scores[chosen][half_b]))
    fixed.append(average_precision_score(truth[half_b], may_scores["logistic, C=1"][half_b]))
print(f"reported (best of 10 on half A): {np.mean(reported):.3f}")
print(f"the same setting on half B:      {np.mean(on_new):.3f}")
print(f"logistic, C=1 on half B:         {np.mean(fixed):.3f}")

> **Check.** You should see `reported (best of 10 on half A): 0.411`, `the same setting on half B: 0.389` and `logistic, C=1 on half B: 0.399`.

The best of 10 on half A looks 0.022 better than it is on new tickets: about three CV standard errors. And the setting chosen this way does no better on new tickets than logistic regression with C = 1, chosen in advance. **When you choose on a set, its score is no longer an honest estimate.** That is why the test set is used once, after every choice.

## 8. Change one thing: rank by ROC AUC

> **Your turn.** In the next cell, change `scoring_name` to `"roc_auc"`, and run the cell (about 15 seconds). Then answer: is `logistic, C=1` within one standard error of the best setting? Set `within` to `True` or `False`, run the cell again, and run the check cell.

In [ ]:
scoring_name = "roc_auc"
search_auc = GridSearchCV(build_pipeline(), grid, cv=cv, scoring=scoring_name, n_jobs=-1)
search_auc.fit(train[FEATURES], train[TARGET])
auc = pd.DataFrame(search_auc.cv_results_)
auc["setting"] = auc["params"].map(describe)
auc["se"] = auc["std_test_score"] / 5 ** 0.5
print(auc[["setting", "mean_test_score", "se", "rank_test_score"]].sort_values("rank_test_score").head(4).round(4).to_string(index=False))
top = auc.sort_values("rank_test_score").iloc[0]
within = bool(auc.loc[auc["setting"] == "logistic, C=1", "mean_test_score"].iloc[0] >= top["mean_test_score"] - top["se"])
print(within)

In [ ]:
expect_hash('your scoring and answer', (scoring_name, within), 'ccd623affeff63dc')

With ROC AUC, the same boosting setting is first (0.7719, standard error 0.0026), and `logistic, C=1` is 0.7696: 0.0023 lower, less than one standard error. The decision does not change. A different metric can change the ranking, so choose the metric before the search (Lesson 1), not after.

## Next

You chose the model with CV on train, and you looked at the test set once. Lesson 3 explains the threshold that `train.py` chose, and checks whether the scores can be read as chances. The [lesson page](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/) has the workflow diagram and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Thresholds and calibration](https://learning.nextia-ai.com/courses/ml/m04/thresholds-and-calibration/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m04/validation-strategy/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.